# 05 · Publication exports

**Scientific question.** Can every manuscript figure, table and quoted number be regenerated from configuration and source alone?

**Scope.** Runs the full reproduction pipeline and inspects what it produced.

**Inputs.** `configs/{PROFILE}.yaml` (loaded below). No other notebook needs to have
been run first: this notebook imports everything it needs from
`src/boundary_aware_dynamics` and holds no state from any other notebook.

**Expected outputs.** Everything under `results/<profile>/`: figures, tables, figure manifest, key-results JSON, LaTeX value macros and a provenance record.

**Approximate runtime.** about 1 minute on `smoke`; several minutes on `paper` on the `smoke` profile.

**Method.** Delegates to `scripts/reproduce.py` so that the notebook and the command line cannot diverge, then verifies the outputs.

**Assumptions.** Only validated results are exported: the notebook does not compute anything the package has not already been tested on.

**References.** See `references/references.bib` and `docs/SCIENTIFIC_METHOD.md`.

**What this notebook does _not_ establish.** Nothing scientific is established here. This notebook is an export and verification step.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

from boundary_aware_dynamics.config import load_config
from boundary_aware_dynamics import plotting

PROFILE = os.environ.get("BAD_PROFILE", "smoke")   # "paper" for manuscript numbers
                                                   # (scripts/execute_notebooks.py sets this)
config = load_config(ROOT / "configs" / f"{PROFILE}.yaml")
plotting.apply_style("preview")
print(f"profile={config.profile}  config_hash={config.config_hash}")

## Run the pipeline

The same entry point as `python scripts/reproduce.py --profile <profile>`, so there is one implementation rather than two.

In [ ]:
import subprocess, sys
completed = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "reproduce.py"), "--profile", PROFILE],
    cwd=ROOT, capture_output=True, text=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr[-3000:])
    raise RuntimeError("reproduce.py failed")

## What was produced

In [ ]:
import pandas as pd, json
out = ROOT / config.output_root
for sub in ("figures", "tables", "metadata"):
    names = sorted(p.name for p in (out / sub).iterdir())
    print(f"{sub} ({len(names)}):")
    for n in names:
        print("   ", n)

## Figure manifest

Every figure carries its source data, configuration hash, dimensions, caption and the command that generated it.

In [ ]:
manifest = pd.read_csv(out / "metadata" / "figure_manifest.csv")
display(manifest[["figure_id", "filename", "source_data", "config_hash", "width_mm"]])
print("\nevery figure has a caption:", manifest["caption"].notna().all())

## Key results and the LaTeX macros

Manuscript numbers are generated, never typed by hand.

In [ ]:
key = json.loads((out / "metadata" / "key_results.json").read_text())
print(json.dumps({k: v for k, v in key.items() if k != "measurement"}, indent=2)[:2000])
print("\n--- paper_values.tex ---")
print((out / "metadata" / "paper_values.tex").read_text())

## Provenance and staleness

A result file existing is not evidence that it is current.

In [ ]:
from boundary_aware_dynamics.provenance import read_provenance, collect_provenance, check_staleness
stored = read_provenance(out / "metadata")
status = check_staleness(stored, collect_provenance(config))
print(json.dumps(status, indent=2))
print("\nreproducible artefact (clean tree, known commit):", stored.is_reproducible_artefact)
if stored.git_dirty:
    print("results were produced from a dirty working tree and are labelled as such")

## Verification

In [ ]:
completed = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "verify_results.py"),
     "--profile", PROFILE, "--skip-tests"],
    cwd=ROOT, capture_output=True, text=True)
print(completed.stdout[-4000:])

## Summary

**Main findings.** The full set of figures, tables and metadata regenerates from configuration and source alone, with a provenance record attached and manuscript numbers emitted as LaTeX macros.

**Validation checks performed.** Presence and schema of every required table; a caption for every figure; provenance staleness; output-hash verification.

**Limitations.** The `smoke` profile is for checking the pipeline, not for manuscript numbers — set `PROFILE = \"paper\"` for those.

**Generated files.** Everything under `results/<profile>/`.

**Relationship to the manuscript.** `results/paper/metadata/paper_values.tex` is intended to be `\\input` directly by the manuscript.

**Next.** See `docs/LOCAL_RELEASE_CHECKLIST.md` for the manual items that remain before any release.